# Bab 2 — Alur Kerja Pra-Model dan Pra-pemrosesan Data

**Buku:** *scikit-learn Cookbook, Third Edition* — John Sukup (Packt, 2025), Bab 2
**Sumber kode:** [PacktPublishing/Scikit-Learn-Cookbook-Third-Edition](https://github.com/PacktPublishing/Scikit-Learn-Cookbook-Third-Edition) (`Chapter_2`)
**Direproduksi dan dijelaskan oleh:** Tarannia Raisha Putri — NIM 101032300231 — S1 Teknik Komputer, Telkom University

**Resep yang dibahas**

| # | Resep | Alat utama |
|---|--------|-----------|
| 2.1 | Menangani Data Hilang (*Handling Missing Data*) | `SimpleImputer`, `KNNImputer`, `IterativeImputer` |
| 2.2 | Teknik Penskalaan (*Scaling Techniques*) | `StandardScaler`, `MinMaxScaler`, `Normalizer` |
| 2.3 | Encoding Variabel Kategorikal | `OneHotEncoder`, `LabelEncoder`, `ColumnTransformer` |
| 2.4 | Pengantar *Pipeline* | `Pipeline`, `train_test_split`, `set_config` |
| 2.5 | Rekayasa Fitur (*Feature Engineering*) | `PolynomialFeatures`, `KBinsDiscretizer`, `RFE`, `SelectFromModel` |
| 2.6 | Latihan Praktis Pra-pemrosesan Data | `Pipeline` + `RandomForestRegressor` pada California Housing |

> Nomor resep ditambahkan di sini untuk memudahkan navigasi; buku aslinya tidak menomori resep. Resep 2.6 memakai solusi resmi dari notebook *Exercise Solution* di repositori.

## Gambaran Umum Bab

Sebelum model apa pun bisa belajar, tabel mentah harus diubah menjadi matriks numerik yang bersih: tanpa nilai kosong, dengan skala fitur yang sebanding, dan dengan kategori yang dinyatakan sebagai angka. Bab ini membahas tahap "pra-model" tersebut. Dimulai dari **imputasi** (mengisi nilai yang hilang), lalu **penskalaan** (menyamakan rentang fitur), kemudian **encoding** (mengubah kategori teks menjadi angka), dan akhirnya menunjukkan cara merangkai semua langkah itu menjadi satu objek yang dapat direproduksi, yaitu **`Pipeline`**. Bab ditutup dengan **rekayasa dan seleksi fitur**: membuat fitur baru (polinomial, *bin*) dan membuang fitur yang tidak membantu (RFE, `SelectFromModel`).

Dalam alur kerja ML secara keseluruhan (*pengumpulan data → pra-pemrosesan → pemodelan → evaluasi → deployment*), bab ini berada tepat setelah pengumpulan data. Hampir semua bab berikutnya bergantung pada apa yang dibangun di sini: PCA dan KNN (Bab 3–4) hanya bermakna pada data yang sudah diskalakan, model linear teregularisasi (Bab 5) mengasumsikan skala fitur yang sebanding, dan setiap model yang di-*tuning* di buku ini dibungkus dalam `Pipeline` agar pra-pemrosesan hanya dipelajari dari data latih.

## Persiapan

In [1]:
# Pustaka inti yang dipakai di sepanjang bab ini
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn

# Satu seed global agar setiap eksekusi notebook menghasilkan angka yang sama.
# (Sel-sel buku tetap memakai nilai random_state masing-masing.)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Catat lingkungan persis yang menghasilkan output di bawah
print(f"Python       : {sys.version.split()[0]}")
print(f"numpy        : {np.__version__}")
print(f"pandas       : {pd.__version__}")
print(f"scikit-learn : {sklearn.__version__}")
print(f"matplotlib   : {matplotlib.__version__}")

Python       : 3.13.5
numpy        : 2.1.3
pandas       : 2.2.3
scikit-learn : 1.5.2
matplotlib   : 3.9.4


## Resep 2.1 — Menangani Data Hilang

**Tujuan.** Sebagian besar estimator scikit-learn menolak input yang mengandung `NaN`. Resep ini mengisi kekosongan dengan tiga strategi yang makin canggih: satu statistik per kolom, nilai dari baris-baris yang mirip, dan model yang memprediksi setiap fitur dari fitur lainnya.

### Teori

**Intuisi.** Membuang setiap baris yang memiliki sel kosong bisa menghilangkan sebagian besar data: dengan 10 fitur dan peluang kosong 20 % per sel, hanya sekitar $0{,}8^{10}\approx 11\%$ baris yang tersisa. Imputasi mempertahankan semua baris dan mengganti setiap kekosongan dengan nilai yang *masuk akal*. Ketiga imputer berbeda dalam seberapa banyak struktur data yang dipakai untuk menentukan nilai "masuk akal" itu:

* **`SimpleImputer`** melihat satu kolom saja dan memakai satu ringkasan (rata-rata, median, modus, atau konstanta).
* **`KNNImputer`** melihat baris secara utuh dan menyalin informasi dari baris lengkap yang paling mirip.
* **`IterativeImputer`** memperlakukan setiap fitur yang tidak lengkap sebagai target regresi yang diprediksi dari fitur-fitur lain.

**Matematika.** Misalkan $x_{ij}$ adalah nilai fitur $j$ untuk sampel $i$, dan $O_j$ adalah himpunan baris tempat fitur $j$ teramati.

* *Imputasi rata-rata:* $\hat{x}_{ij} = \bar{x}_j = \dfrac{1}{|O_j|}\sum_{k\in O_j} x_{kj}$. Strategi median dan *most frequent* mengganti $\bar{x}_j$ dengan median atau modus dari $\{x_{kj}: k\in O_j\}$.
* *Imputasi KNN:* jarak antarbaris dihitung hanya pada koordinat yang ada di kedua baris, lalu diskalakan ulang ke dimensi penuh $p$ (metrik `nan_euclidean`):
$$d(\mathbf{a},\mathbf{b}) = \sqrt{\frac{p}{|C|}\sum_{j\in C}(a_j-b_j)^2},$$
dengan $C$ adalah himpunan fitur yang teramati pada $\mathbf{a}$ maupun $\mathbf{b}$. Nilai hilang $\hat{x}_{ij}$ adalah rata-rata fitur $j$ dari $k$ baris terdekat yang memiliki fitur $j$ (bobot seragam secara default).
* *Imputasi iteratif (gaya MICE):* mulai dari pengisian sederhana (rata-rata), lalu bergilir ke setiap fitur. Untuk fitur $j$, latih regresor $f_j$ pada baris yang fitur $j$-nya teramati, $x_{j} \approx f_j(\mathbf{x}_{-j})$, dengan $\mathbf{x}_{-j}$ adalah semua fitur lain, lalu timpa nilai yang hilang dengan $f_j(\mathbf{x}_{-j})$. Satu putaran ke semua fitur disebut satu ronde; ronde diulang sampai perubahan terbesar berada di bawah toleransi atau `max_iter` tercapai.

**Cara scikit-learn mengimplementasikannya.**

| Kelas | Parameter kunci (default) | `fit` mempelajari | `transform` melakukan |
|---|---|---|---|
| `SimpleImputer` | `strategy="mean"`, `missing_values=np.nan`, `fill_value=None`, `add_indicator=False` | satu statistik per kolom → `statistics_` | mengganti setiap `NaN` dengan statistik kolomnya |
| `KNNImputer` | `n_neighbors=5`, `weights="uniform"`, `metric="nan_euclidean"` | menyimpan matriks data latih | mencari tetangga di data yang disimpan dan merata-ratakan nilainya |
| `IterativeImputer` | `estimator=BayesianRidge()`, `max_iter=10`, `tol=1e-3`, `initial_strategy="mean"`, `imputation_order="ascending"` | satu regresor per fitur per ronde | menjalankan ulang regresor yang telah dipelajari pada data baru |

`IterativeImputer` masih berstatus eksperimental, sehingga harus diaktifkan dulu dengan `from sklearn.experimental import enable_iterative_imputer` sebelum bisa diimpor.

**Asumsi, kelebihan, keterbatasan.**

* Ketiganya mengasumsikan data hilang *sepenuhnya acak* (MCAR) atau *acak bila diketahui fitur yang teramati* (MAR). Jika fakta bahwa suatu nilai hilang membawa informasi (MNAR, misalnya orang berpenghasilan tinggi enggan mengisi pertanyaan penghasilan), tambahkan `add_indicator=True` agar model bisa "melihat" pola kehilangannya.
* Imputasi rata-rata/median cepat dan stabil, tetapi memperkecil variansi setiap kolom dan melemahkan korelasi, karena semua kekosongan diisi nilai yang sama.
* Imputasi KNN mempertahankan struktur lokal tetapi berbasis jarak. Fitur sebaiknya berada pada skala yang mirip, dan biayanya naik seiring jumlah baris yang disimpan.
* Imputasi iteratif dapat memulihkan korelasi antarfitur, tetapi lebih lambat dan hanya lebih baik daripada rata-rata jika fitur-fiturnya memang saling memprediksi.
* Imputer mana pun yang dipilih, **fit hanya pada data latih** (idealnya di dalam `Pipeline`) agar statistik dari data uji tidak bocor ke pelatihan.

### Kode

In [2]:
# Impor untuk membangun dataset mainan
import numpy as np
import pandas as pd

# Seed tetap agar tabel acak (beserta kekosongannya) dapat direproduksi
np.random.seed(2024)
n_samples = 20
n_features = 10

# Sepuluh kolom berisi nilai seragam di [0, 100)
data = {
    f"Feature{i+1}": np.random.uniform(0, 100, n_samples) for i in range(n_features)
}

# Bungkus dictionary dalam DataFrame agar mudah dibaca
df = pd.DataFrame(data)

# Kosongkan sekitar 20% sel di setiap kolom secara acak
for column in df.columns:
    mask = np.random.random(n_samples) < 0.2
    df.loc[mask, column] = np.nan

# Tampilkan tabel, termasuk NaN
display(df)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,NaN,42.009814,34.680397,49.962259,NaN,NaN,89.954588,41.152421,NaN
1,69.910875,9.554215,6.436369,31.287816,37.966499,NaN,82.005649,NaN,75.797616,91.382492
2,NaN,96.090974,59.643269,84.710402,NaN,84.109027,NaN,70.288128,1.778343,4.117690
3,NaN,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,NaN,NaN,80.077973
4,20.501895,NaN,89.248639,67.655865,58.635861,78.225721,60.911562,NaN,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,NaN,19.703051,34.423301,93.399494,72.206680,12.640276
6,72.724014,79.792340,50.239523,55.921377,6.191019,NaN,22.966899,21.049022,57.358544,14.302591
7,NaN,NaN,89.538184,69.451294,NaN,47.885551,NaN,33.620401,99.685711,6.683138
8,47.384570,NaN,25.592093,82.419730,73.414540,61.663700,29.172571,65.946718,61.005155,34.052747
9,44.829582,38.165095,NaN,31.142866,28.865545,NaN,41.004459,41.460336,50.236236,NaN


In [3]:
# Audit cepat: berapa nilai yang hilang per kolom? (tambahan)
df.isna().sum().to_frame("jumlah_hilang").T

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
jumlah_hilang,3,5,4,4,5,5,7,6,2,3


**Interpretasi output.** Tabel mainan berisi 20 baris × 10 kolom nilai seragam antara 0 dan 100. Baris audit menunjukkan sebaran kekosongan: 44 dari 200 sel (22 %) hilang, mulai dari 7 di `Feature7` hingga hanya 2 di `Feature9`. Karena setiap kolom dibangkitkan secara independen, fitur-fitur ini **tidak membawa informasi satu sama lain**. Hal ini akan penting saat membandingkan ketiga imputer.

In [4]:
from sklearn.impute import SimpleImputer

# Imputasi rata-rata per kolom (median / most_frequent bekerja dengan cara yang sama)
imputer = SimpleImputer(strategy="mean")

# fit mempelajari satu rata-rata per kolom; transform menuliskannya ke setiap NaN
imputed_data = imputer.fit_transform(df)
imputed_df = pd.DataFrame(imputed_data, columns=df.columns)
imputed_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,53.864661,42.009814,34.680397,49.962259,57.822964,51.145011,89.954588,41.152421,48.400044
1,69.910875,9.554215,6.436369,31.287816,37.966499,57.822964,82.005649,50.715003,75.797616,91.382492
2,52.558605,96.090974,59.643269,84.710402,46.055883,84.109027,51.145011,70.288128,1.778343,4.117690
3,52.558605,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,50.715003,60.616723,80.077973
4,20.501895,53.864661,89.248639,67.655865,58.635861,78.225721,60.911562,50.715003,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,46.055883,19.703051,34.423301,93.399494,72.206680,12.640276
6,72.724014,79.792340,50.239523,55.921377,6.191019,57.822964,22.966899,21.049022,57.358544,14.302591
7,52.558605,53.864661,89.538184,69.451294,46.055883,47.885551,51.145011,33.620401,99.685711,6.683138
8,47.384570,53.864661,25.592093,82.419730,73.414540,61.663700,29.172571,65.946718,61.005155,34.052747
9,44.829582,38.165095,52.897507,31.142866,28.865545,57.822964,41.004459,41.460336,50.236236,48.400044


**Interpretasi output.** Setiap kekosongan dalam satu kolom kini berisi angka yang *sama*, yaitu rata-rata nilai teramati kolom tersebut. Contohnya, ketiga sel `Feature1` yang hilang (baris 2, 3, dan 7) menjadi **52.558605**, dan setiap sel `Feature7` yang hilang menjadi **51.145011**. Nilai yang teramati tidak berubah. Konstanta berulang ini adalah kelemahan imputasi rata-rata yang terlihat jelas: nilai menumpuk di tengah distribusi dan menurunkan variansi kolom.

In [5]:
from sklearn.impute import KNNImputer

# Isi setiap kekosongan dengan rata-rata dari 2 baris yang paling mirip
knn_imputer = KNNImputer(n_neighbors=2)

knn_imputed_data = knn_imputer.fit_transform(df)
knn_imputed_df = pd.DataFrame(knn_imputed_data, columns=df.columns)
knn_imputed_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,48.954043,42.009814,34.680397,49.962259,93.910386,52.549271,89.954588,41.152421,59.833678
1,69.910875,9.554215,6.436369,31.287816,37.966499,86.793468,82.005649,45.416191,75.797616,91.382492
2,67.752344,96.090974,59.643269,84.710402,73.338309,84.109027,59.012876,70.288128,1.778343,4.117690
3,47.880864,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,64.510281,39.726833,80.077973
4,20.501895,31.670912,89.248639,67.655865,58.635861,78.225721,60.911562,25.903612,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,59.716773,19.703051,34.423301,93.399494,72.206680,12.640276
6,72.724014,79.792340,50.239523,55.921377,6.191019,40.482911,22.966899,21.049022,57.358544,14.302591
7,47.629715,77.843853,89.538184,69.451294,32.753328,47.885551,55.683434,33.620401,99.685711,6.683138
8,47.384570,35.180639,25.592093,82.419730,73.414540,61.663700,29.172571,65.946718,61.005155,34.052747
9,44.829582,38.165095,29.188271,31.142866,28.865545,59.674651,41.004459,41.460336,50.236236,35.000820


**Interpretasi output.** Nilai hasil imputasi KNN kini berbeda dari baris ke baris karena setiap nilai berasal dari dua tetangga terdekat baris itu sendiri. `Feature1` yang hilang di baris 2 menjadi **67.752344**, di baris 3 menjadi **47.880864**, dan `Feature3` yang hilang di baris 10 menjadi **88.326309**, semuanya jauh dari rata-rata kolom. Dengan hanya dua tetangga, hasil pengisian cukup "berisik". Setiap nilai hanyalah rata-rata dua observasi yang dipilih berdasarkan jarak yang dihitung pada baris yang sebagian kosong, sehingga pada data tanpa struktur ini hasilnya lebih mirip pengambilan acak dari kolom daripada estimasi yang terinformasi.

In [6]:
# IterativeImputer masih eksperimental, jadi harus diaktifkan secara eksplisit terlebih dahulu
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Regresi bergiliran: setiap fitur yang tidak lengkap diprediksi dari semua fitur lain
iterative_imputer = IterativeImputer()

iterative_imputed_data = iterative_imputer.fit_transform(df)
iterative_imputed_df = pd.DataFrame(iterative_imputed_data, columns=df.columns)
iterative_imputed_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,54.365008,42.009814,34.680397,49.962259,58.680582,51.178102,89.954588,41.152421,48.292275
1,69.910875,9.554215,6.436369,31.287816,37.966499,60.070634,82.005649,50.710323,75.797616,91.382492
2,52.539676,96.090974,59.643269,84.710402,46.121290,84.109027,51.155371,70.288128,1.778343,4.117690
3,52.630591,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,50.680265,50.650877,80.077973
4,20.501895,53.198954,89.248639,67.655865,58.635861,78.225721,60.911562,50.691674,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,46.156310,19.703051,34.423301,93.399494,72.206680,12.640276
6,72.724014,79.792340,50.239523,55.921377,6.191019,55.923578,22.966899,21.049022,57.358544,14.302591
7,52.558131,53.942885,89.538184,69.451294,46.070477,47.885551,50.969808,33.620401,99.685711,6.683138
8,47.384570,54.204978,25.592093,82.419730,73.414540,61.663700,29.172571,65.946718,61.005155,34.052747
9,44.829582,38.165095,52.882614,31.142866,28.865545,55.978300,41.004459,41.460336,50.236236,48.429901


**Interpretasi output.** Hasil pengisian iteratif berada **sangat dekat dengan rata-rata kolom** (tiga kekosongan `Feature1` menjadi 52.539676, 52.630591, dan 52.558131, bukan tepat 52.558605). Inilah yang diprediksi teori untuk dataset ini: kolom-kolomnya independen, sehingga regresor default `BayesianRidge` mempelajari koefisien yang mendekati nol dan prediksinya runtuh ke arah intersep, yaitu rata-rata kolom. Pada data nyata dengan fitur yang saling berkorelasi, hasil pengisian iteratif akan bergeser dari rata-rata mengikuti korelasi tersebut.

### Poin Penting
* `SimpleImputer` menuliskan satu konstanta per kolom. Cepat dan stabil, tetapi meratakan distribusi.
* `KNNImputer` meminjam nilai dari baris yang mirip. Ia membutuhkan fitur yang sudah diskalakan dan `n_neighbors` yang wajar; dengan tetangga yang sangat sedikit hasilnya berisik.
* `IterativeImputer` memodelkan setiap fitur dari fitur lainnya dan hanya membantu jika fitur-fiturnya memang berkorelasi. Di sini hasilnya tereduksi menjadi imputasi rata-rata.
* Imputer adalah transformer yang belajar dari data: fit hanya pada data latih, idealnya di dalam `Pipeline`.

## Resep 2.2 — Teknik Penskalaan

**Tujuan.** Menempatkan fitur pada rentang numerik yang sebanding agar tidak ada fitur yang mendominasi perhitungan jarak atau pembaruan gradien hanya karena satuannya. Resep ini menerapkan tiga transformer pada tabel hasil imputasi iteratif dari Resep 2.1.

### Teori

**Intuisi.** Model KNN yang mengukur jarak antarrumah berdasarkan *luas dalam m²* (ratusan) dan *jumlah kamar* (satuan) praktis mengabaikan jumlah kamar. Penskalaan memperbaiki masalah satuan ini. Scaler per kolom (*standardisasi*, *min-max*) menyesuaikan setiap **fitur**, sedangkan `Normalizer` menyesuaikan setiap **sampel (baris)** agar panjangnya satu. Ini operasi yang berbeda, dipakai ketika yang penting hanyalah arah sebuah vektor.

**Matematika.** Untuk fitur $j$ dengan rata-rata latih $\mu_j$, simpangan baku $\sigma_j$, minimum $m_j$, dan maksimum $M_j$:

* **Standardisasi (`StandardScaler`):** $z_{ij} = \dfrac{x_{ij}-\mu_j}{\sigma_j}$, dengan $\sigma_j = \sqrt{\tfrac{1}{n}\sum_i (x_{ij}-\mu_j)^2}$ (simpangan baku populasi, `ddof=0`). Setiap kolom berakhir dengan rata-rata 0 dan variansi 1.
* **Penskalaan min-max (`MinMaxScaler`):** $x'_{ij} = \dfrac{x_{ij}-m_j}{M_j-m_j}\,(b-a)+a$, dengan $(a,b)$ adalah `feature_range`, default $(0,1)$.
* **Normalisasi baris (`Normalizer`):** $\mathbf{x}'_i = \dfrac{\mathbf{x}_i}{\lVert\mathbf{x}_i\rVert_p}$, dengan $\lVert\mathbf{x}_i\rVert_2=\sqrt{\sum_j x_{ij}^2}$ untuk default `norm="l2"` ($p$ juga bisa 1 atau $\infty$ lewat `"l1"`/`"max"`).

**Cara scikit-learn mengimplementasikannya.**

* `StandardScaler(with_mean=True, with_std=True)`: `fit` menyimpan `mean_`, `var_`, dan `scale_`, lalu `transform` menerapkan $(x-\mu)/\sigma$.
* `MinMaxScaler(feature_range=(0, 1), clip=False)`: `fit` menyimpan `data_min_` dan `data_max_`. Karena `clip=False`, data baru di luar rentang latih akan dipetakan ke luar $[a,b]$.
* `Normalizer(norm="l2")` bersifat **stateless**: `fit` tidak mempelajari apa pun, dan `transform` menskalakan setiap baris secara independen, sehingga kebocoran latih/uji tidak mungkin terjadi.

**Asumsi, kelebihan, keterbatasan.**

* Model berbasis jarak (KNN, SVM dengan kernel RBF, K-means), berbasis gradien (regresi logistik, jaringan saraf), dan model berpenalti (Ridge/Lasso) sensitif terhadap skala. Model berbasis pohon tidak.
* `StandardScaler` memakai rata-rata dan simpangan baku yang dapat terdistorsi oleh pencilan (*outlier*). `MinMaxScaler` lebih rapuh lagi, karena satu nilai ekstrem menekan semua nilai lainnya. `RobustScaler` (median dan IQR) adalah alternatif yang tahan pencilan.
* `Normalizer` cocok ketika *arah* vektor sampel lebih penting daripada besarnya, seperti vektor teks TF-IDF atau *cosine similarity*. Ia **bukan** pengganti penskalaan fitur.

### Kode

In [7]:
from sklearn.preprocessing import StandardScaler

# Pusatkan setiap kolom di 0 dan beri variansi satu
scaler = StandardScaler()

scaled_data = scaler.fit_transform(iterative_imputed_df)
scaled_df = pd.DataFrame(scaled_data, columns=iterative_imputed_df.columns)
scaled_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.271931,0.019495,-0.373057,-0.894258,0.188761,0.031844,0.001905,1.683800,-0.780898,-0.003574
1,0.756048,-1.874939,-1.592196,-1.043864,-0.391744,0.085949,1.426871,-0.000534,0.652484,1.429056
2,-0.000939,1.783513,0.231259,1.311954,0.002887,1.021588,0.000854,0.839730,-2.409929,-1.472256
3,0.003022,-1.214477,1.056818,1.458037,-1.411839,1.531340,1.551267,-0.001824,-0.387917,1.053212
4,-1.397054,-0.029802,1.245866,0.559887,0.608499,0.792594,0.451822,-0.001334,0.210478,1.686279
5,-1.828276,0.969036,-1.125549,-2.186891,0.004582,-1.485267,-0.772566,1.831653,0.503915,-1.188904
6,0.878636,1.094467,-0.091017,0.042422,-1.929442,-0.075466,-1.302124,-1.273574,-0.110399,-1.133636
7,-0.000135,0.001649,1.255789,0.639061,0.000428,-0.388327,-0.007723,-0.734020,1.640810,-1.386962
8,-0.225584,0.012729,-0.935710,1.210940,1.323678,0.147955,-1.015275,0.653400,0.040472,-0.476999
9,-0.336923,-0.665377,-0.000435,-1.050256,-0.832163,-0.073336,-0.468360,-0.397536,-0.405072,0.001002


In [8]:
# Cek kewajaran: rata-rata kolom harus ~0 dan simpangan baku (populasi) ~1 (tambahan)
scaled_df.agg(["mean", "std"]).round(3)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
mean,-0.000,0.000,0.000,-0.000,0.000,-0.000,-0.000,-0.000,0.000,-0.000
std,1.026,1.026,1.026,1.026,1.026,1.026,1.026,1.026,1.026,1.026


**Interpretasi output.** Setelah standardisasi, sebagian besar nilai berada di antara sekitar −2,4 dan +2,0. Pengecekan mengonfirmasi setiap kolom kini memiliki rata-rata 0.000. Simpangan baku yang tampil adalah 1.026, bukan 1, karena pandas memakai rumus sampel ($n-1$), sedangkan `StandardScaler` membagi dengan $n$: $\sqrt{20/19}\approx1{,}026$. Perhatikan bahwa sel hasil imputasi menjadi **hampir tepat 0** (misalnya `Feature1` baris 2 = −0.000939, `Feature5` baris 7 = 0.000428). Sel-sel itu diisi dengan nilai dekat rata-rata, dan standardisasi memetakan rata-rata ke nol.

In [9]:
from sklearn.preprocessing import MinMaxScaler

# Petakan setiap kolom secara linear ke [0, 1]
minmax_scaler = MinMaxScaler()

minmax_scaled_data = minmax_scaler.fit_transform(iterative_imputed_df)
minmax_scaled_df = pd.DataFrame(
    minmax_scaled_data, columns=iterative_imputed_df.columns
)
minmax_scaled_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.603506,0.517824,0.380540,0.354639,0.569020,0.555896,0.435006,0.962767,0.402156,0.464988
1,0.721357,0.000000,0.000369,0.313594,0.413077,0.571920,0.833074,0.538604,0.756013,0.918562
2,0.537080,1.000000,0.568987,0.959922,0.519088,0.849027,0.434713,0.750206,0.000000,0.000000
3,0.538045,0.180530,0.826426,1.000000,0.139045,1.000000,0.867825,0.538279,0.499171,0.799569
4,0.197218,0.504349,0.885377,0.753589,0.681776,0.781206,0.560692,0.538402,0.646896,1.000000
5,0.092244,0.777371,0.145886,0.000000,0.519543,0.106575,0.218656,1.000000,0.719336,0.089710
6,0.751199,0.811657,0.468490,0.611621,0.000000,0.524114,0.070723,0.218016,0.567681,0.107208
7,0.537276,0.512946,0.888472,0.775311,0.518428,0.431454,0.432317,0.353891,1.000000,0.027004
8,0.482394,0.515975,0.205085,0.932208,0.873897,0.590284,0.150855,0.703283,0.604927,0.315101
9,0.455290,0.330621,0.496737,0.311840,0.294766,0.524745,0.303637,0.438627,0.494936,0.466437


**Interpretasi output.** Setiap kolom kini tepat mencakup $[0,1]$: setiap kolom memiliki satu baris bernilai `0.000000` (minimumnya) dan satu bernilai `1.000000` (maksimumnya). Misalnya, baris 13 adalah maksimum `Feature1` dan baris 10 minimumnya. Jarak relatif di dalam setiap kolom tetap terjaga, karena penskalaan min-max adalah pemetaan linear.

In [10]:
from sklearn.preprocessing import Normalizer

# Skalakan setiap BARIS agar panjang Euclidean-nya satu (tidak ada statistik yang dipelajari)
normalizer = Normalizer()

normalized_data = normalizer.fit_transform(iterative_imputed_df)
normalized_df = pd.DataFrame(normalized_data, columns=iterative_imputed_df.columns)
normalized_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.339168,0.313579,0.242313,0.200037,0.288183,0.338471,0.295196,0.518860,0.237368,0.278551
1,0.376706,0.051482,0.034682,0.168591,0.204578,0.323683,0.441878,0.273247,0.408426,0.492404
2,0.264336,0.483450,0.300075,0.426192,0.232044,0.423166,0.257371,0.353631,0.008947,0.020717
3,0.243762,0.116607,0.387811,0.407684,0.078213,0.450213,0.392278,0.234729,0.234592,0.370886
4,0.095909,0.248868,0.417511,0.316499,0.274302,0.365945,0.284948,0.237139,0.304609,0.463686
5,0.068115,0.493382,0.128781,0.034471,0.296421,0.126535,0.221071,0.599823,0.463720,0.081177
6,0.460521,0.505281,0.318139,0.354119,0.039204,0.354133,0.145437,0.133292,0.363220,0.090570
7,0.274582,0.281816,0.467778,0.362837,0.240688,0.250170,0.266284,0.175644,0.520792,0.034915
8,0.265283,0.303467,0.143277,0.461427,0.411012,0.345225,0.163323,0.369203,0.341538,0.190645
9,0.321287,0.273524,0.379002,0.223196,0.206875,0.401188,0.293873,0.297140,0.360036,0.347090


In [11]:
# Setiap baris sekarang harus memiliki norma L2 sama dengan 1 (tambahan)
np.linalg.norm(normalized_df.values, axis=1).round(6)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

**Interpretasi output.** Berbeda dengan dua scaler sebelumnya, `Normalizer` bekerja di sepanjang setiap baris: pengecekan menunjukkan bahwa setiap dari 20 baris kini memiliki norma Euclidean **1.0**. Nilai dalam satu baris sekarang merupakan *proporsi* dari panjang baris tersebut (entri terbesar baris 0, `Feature8`, bernilai 0.519). Rata-rata dan variansi kolom sama sekali tidak dikendalikan, itulah sebabnya normalisasi baris dipakai untuk kemiripan berbasis arah, bukan sebagai penskalaan fitur umum.

### Poin Penting
* `StandardScaler` → rata-rata 0 dan variansi satu per kolom. Pilihan default untuk model linear, SVM, dan PCA.
* `MinMaxScaler` → setiap kolom dipetakan ke rentang tetap. Mudah ditafsirkan tetapi sensitif terhadap pencilan.
* `Normalizer` → setiap **baris** memiliki norma satu. Gunakan untuk kemiripan bergaya kosinus, bukan sebagai penskalaan fitur.
* Scaler per kolom mempelajari statistik di `fit`, sehingga aturan kebocoran yang sama dengan imputer berlaku.

## Resep 2.3 — Encoding Variabel Kategorikal

**Tujuan.** Mengubah kategori teks (departemen, jabatan, kota) menjadi angka yang dapat diproses estimator, dan menerapkan pra-pemrosesan berbeda pada kolom numerik dan kategorikal dalam satu langkah.

### Teori

**Intuisi.** Model menghitung dengan angka, sehingga "SF" atau "Manager" harus di-*encode*. Bahayanya adalah **menciptakan urutan yang sebenarnya tidak ada**. Mengodekan kota sebagai 0, 1, 2, 3 memberi tahu model linear bahwa SF (3) "tiga kali" CHI (0). *One-hot encoding* menghindari hal ini dengan memberi setiap kategori kolom 0/1 sendiri.

**Matematika.**

* **One-hot:** fitur dengan kategori $\{c_1,\dots,c_K\}$ memetakan kategori $c_k$ ke vektor satuan $\mathbf{e}_k\in\{0,1\}^K$ (satu angka 1 di posisi $k$). Baris dengan beberapa fitur kategorikal menyambung blok-blok ini, sehingga lebar output adalah $\sum_f K_f$.
* **Label (ordinal) encoding:** $c_k \mapsto k-1 \in \{0,\dots,K-1\}$, dengan kategori diurutkan secara leksikografis.
* **ColumnTransformer:** untuk transformer $T_1,\dots,T_m$ yang bekerja pada subset kolom $S_1,\dots,S_m$, outputnya adalah penyambungan horizontal $[\,T_1(X_{S_1})\;|\;\cdots\;|\;T_m(X_{S_m})\;|\;X_{R}\,]$, dengan $X_R$ adalah kolom sisa yang dipertahankan oleh `remainder="passthrough"`.

**Cara scikit-learn mengimplementasikannya.**

* `OneHotEncoder(categories="auto", drop=None, sparse_output=True, handle_unknown="error")`: `fit` mencatat kategori terurut per kolom di `categories_`, dan `transform` membangun matriks indikator. `sparse_output=False` mengembalikan array padat (argumen ini menggantikan `sparse=` yang lama sejak v1.2). `get_feature_names_out()` menghasilkan nama seperti `Department_A`.
* `LabelEncoder()` dirancang untuk **vektor target 1-D `y`**. `fit` menyimpan `classes_` (label unik terurut), dan `transform` mengembalikan indeksnya. Untuk kolom fitur, alat yang semestinya adalah `OrdinalEncoder`, yang menangani input 2-D. Resep ini tetap melakukan perulangan per kolom untuk menerapkan `LabelEncoder` pada fitur, dengan me-*refit* setiap kali.
* `ColumnTransformer(transformers, remainder="drop", sparse_threshold=0.3)`: setiap triplet `(nama, transformer, kolom)` di-fit pada kolomnya sendiri, output ditumpuk berdampingan, dan bagian yang sudah di-fit dapat diakses lewat `named_transformers_`.

**Asumsi, kelebihan, keterbatasan.**

* One-hot aman untuk data nominal tetapi menjadi sangat lebar untuk fitur berkardinalitas tinggi (ribuan kode pos). Dalam kasus itu pertimbangkan `min_frequency`/`max_categories`, *target encoding*, atau *hashing*.
* Pada model linear tanpa regularisasi, blok one-hot lengkap berkolinear sempurna dengan intersep (*dummy-variable trap*). `drop="first"` menghapus satu kolom per fitur.
* Encoding bilangan bulat cocok untuk kategori yang memang berurutan (Junior < Senior < Manager) atau untuk model pohon, yang hanya memisah berdasarkan ambang batas dan tidak terlalu terganggu oleh urutan sembarang.
* `ColumnTransformer` adalah cara standar untuk menyatukan pra-pemrosesan numerik dan kategorikal secara bebas kebocoran di dalam `Pipeline`.

### Kode

In [12]:
import numpy as np

# 20 karyawan acak dengan tiga fitur nominal
np.random.seed(2024)
categories = ["A", "B", "C", "D"]
categorical_data = pd.DataFrame(
    {
        "Department": np.random.choice(categories, size=20),
        "Position": np.random.choice(["Junior", "Senior", "Manager"], size=20),
        "Location": np.random.choice(["NY", "SF", "LA", "CHI"], size=20),
    }
)

display(categorical_data)

,Department,Position,Location
0,A,Manager,LA
1,C,Manager,LA
2,A,Junior,NY
3,A,Manager,LA
4,D,Manager,NY
5,A,Senior,LA
6,C,Manager,SF
7,D,Manager,LA
8,B,Junior,LA
9,D,Manager,CHI


In [13]:
from sklearn.preprocessing import OneHotEncoder

# Output padat agar hasilnya bisa ditampilkan sebagai DataFrame
onehot_encoder = OneHotEncoder(sparse_output=False)

# Satu kolom indikator per pasangan (fitur, kategori)
onehot_encoded_data = onehot_encoder.fit_transform(categorical_data)
onehot_encoded_df = pd.DataFrame(
    onehot_encoded_data, columns=onehot_encoder.get_feature_names_out()
)
onehot_encoded_df

,Department_A,Department_B,Department_C,Department_D,Position_Junior,Position_Manager,Position_Senior,Location_CHI,Location_LA,Location_NY,Location_SF
0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
5,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
6,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
7,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
8,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
9,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


**Interpretasi output.** Tiga kolom kategorikal menjadi **11 kolom indikator**: 4 departemen + 3 jabatan + 4 lokasi. Setiap baris berisi tepat tiga angka 1, satu per fitur asli. Baris 0 (`A`, `Manager`, `LA`), misalnya, bernilai 1.0 di `Department_A`, `Position_Manager`, dan `Location_LA`, dan 0.0 di tempat lain. Nama kolom berasal dari `get_feature_names_out()`, dengan kategori diurutkan secara alfabetis.

In [14]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

# Kumpulkan kode bilangan bulat kolom demi kolom
label_encoded_df = pd.DataFrame()

# Encoder di-fit ulang pada setiap kolom, sehingga setiap kolom memiliki pemetaan sendiri
for column in categorical_data.columns:
    label_encoded_df[f"{column}_encoded"] = label_encoder.fit_transform(
        categorical_data[column]
    )
label_encoded_df

,Department_encoded,Position_encoded,Location_encoded
0,0,1,1
1,2,1,1
2,0,0,2
3,0,1,1
4,3,1,2
5,0,2,1
6,2,1,3
7,3,1,1
8,1,0,1
9,3,1,0


**Interpretasi output.** Setiap kolom kini menjadi satu kolom bilangan bulat dengan kode yang diberikan secara **alfabetis**: `A→0 … D→3`, `Junior→0, Manager→1, Senior→2`, `CHI→0, LA→1, NY→2, SF→3`. Baris 0 (`A`, `Manager`, `LA`) menjadi `(0, 1, 1)`. Perhatikan bahwa "Manager" berada *di antara* Junior dan Senior, urutan yang berasal dari alfabet, bukan dari makna bisnisnya. Ini menunjukkan mengapa kode bilangan bulat dapat menyesatkan model yang memperlakukan input sebagai besaran kontinu.

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

# Tabel campuran: tiga kolom numerik dan dua kolom kategorikal
np.random.seed(2024)
mixed_data = pd.DataFrame(
    {
        "Age": np.random.randint(25, 65, size=20),
        "Salary": np.round(np.random.normal(60000, 15000, size=20), 2),
        "Experience": np.random.randint(1, 20, size=20),
        "Department": np.random.choice(["IT", "HR", "Sales", "Finance"], size=20),
        "Position": np.random.choice(["Junior", "Senior", "Manager"], size=20),
    }
)

display(mixed_data)

,Age,Salary,Experience,Department,Position
0,33,59420.36,12,Finance,Manager
1,57,82895.92,17,Sales,Junior
2,25,38165.76,16,Finance,Manager
3,52,38242.36,7,IT,Junior
4,61,55088.65,8,Sales,Manager
5,26,78688.88,9,Sales,Senior
6,60,49585.62,14,Finance,Junior
7,35,47992.58,17,HR,Manager
8,27,54833.02,9,Sales,Senior
9,57,93535.56,14,Sales,Senior


In [16]:
# Skalakan kolom numerik, one-hot kolom kategorikal, pertahankan sisanya apa adanya
column_transformer = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ["Age", "Salary", "Experience"]),
        ("cat", OneHotEncoder(), ["Department", "Position"]),
    ],
    remainder="passthrough",
)

transformed_data = column_transformer.fit_transform(mixed_data)

# Susun nama kolom yang mudah dibaca dari sub-transformer yang sudah di-fit
numeric_cols = ["Age_scaled", "Salary_scaled", "Experience_scaled"]
categorical_cols = column_transformer.named_transformers_["cat"].get_feature_names_out(
    ["Department", "Position"]
)

transformed_df = pd.DataFrame(
    transformed_data, columns=numeric_cols + list(categorical_cols)
)
transformed_df

,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager,Position_Senior
0,-1.045349,-0.305043,0.327303,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.727681,1.105091,1.262454,0.0,0.0,0.0,1.0,1.0,0.0,0.0
2,-1.636358,-1.581768,1.075424,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0.358300,-1.577167,-0.607848,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4,1.023186,-0.565241,-0.420818,0.0,0.0,0.0,1.0,0.0,1.0,0.0
5,-1.562482,0.852382,-0.233788,0.0,0.0,0.0,1.0,0.0,0.0,1.0
6,0.949309,-0.895798,0.701364,1.0,0.0,0.0,0.0,1.0,0.0,0.0
7,-0.897596,-0.991489,1.262454,0.0,1.0,0.0,0.0,0.0,1.0,0.0
8,-1.488606,-0.580596,-0.233788,0.0,0.0,0.0,1.0,0.0,0.0,1.0
9,0.727681,1.744195,0.701364,0.0,0.0,0.0,1.0,0.0,0.0,1.0


**Interpretasi output.** Lima kolom mentah menjadi **10 kolom siap-model**: 3 kolom numerik terstandardisasi diikuti 4 indikator `Department_*` dan 3 indikator `Position_*`. Urutan output mengikuti urutan daftar `transformers`. Baris 0 (usia 33, gaji 59.420,36) berada −1,045 simpangan baku di bawah rata-rata usia dan hanya −0,305 simpangan baku di bawah rata-rata gaji. Semua kolom sudah dipakai oleh suatu transformer, sehingga `remainder="passthrough"` tidak meneruskan apa pun. `OneHotEncoder` di sini memakai output *sparse* default, tetapi hasil tumpukannya tetap array padat karena kepadatan keseluruhannya di atas `sparse_threshold=0.3`.

### Poin Penting
* *One-hot encoding* adalah pilihan default yang aman untuk kategori nominal. Lebar = total jumlah kategori.
* `LabelEncoder` ditujukan untuk target. Kode bilangan bulat pada fitur memaksakan urutan alfabetis yang mungkin tidak bermakna.
* `ColumnTransformer` mengarahkan setiap kelompok kolom ke pra-pemrosesan yang tepat dan menyambung hasilnya, sehingga menjadi tulang punggung *pipeline* pra-pemrosesan di dunia nyata.

## Resep 2.4 — Pengantar *Pipeline*

**Tujuan.** Merangkai beberapa langkah pra-pemrosesan (dan opsional satu model di akhir) menjadi satu objek yang di-fit sekali pada data latih lalu diterapkan secara identik pada data uji. Hal ini membuat alur kerja ringkas, dapat direproduksi, dan bebas kebocoran.

### Teori

**Intuisi.** Tanpa *pipeline*, kita harus ingat untuk memanggil `fit_transform` pada data latih dan hanya `transform` pada data uji di setiap langkah, dengan urutan yang benar. `Pipeline` mengodekan urutan itu sekali saja. Ia juga membuat seluruh rantai berperilaku seperti satu estimator, sehingga *cross-validation* dan *grid search* (Bab 4–8) dapat mem-fit ulang pra-pemrosesan di dalam setiap *fold*.

**Matematika.** *Pipeline* dengan transformer $T_1,\dots,T_m$ dan estimator akhir (opsional) $f$ adalah komposisi
$$\hat{y} = f\big(T_m(\cdots T_2(T_1(\mathbf{x}))\cdots)\big).$$
Selama `fit`, setiap $T_k$ di-fit pada output $T_{k-1}$ yang dihitung **hanya dari data latih**, dan $f$ di-fit pada output $T_m$.

**Cara scikit-learn mengimplementasikannya.**

* `Pipeline(steps, memory=None, verbose=False)`, dengan `steps` berupa daftar pasangan `(nama, estimator)`. Semua langkah kecuali yang terakhir harus mengimplementasikan `fit`/`transform`.
* `pipeline.fit(X, y)` memanggil `fit_transform` pada setiap langkah perantara secara berurutan, lalu `fit` pada langkah terakhir.
* `pipeline.transform(X)` / `predict(X)` hanya memanggil `transform` pada langkah perantara (memakai apa yang dipelajari saat `fit`), lalu `transform`/`predict` langkah terakhir.
* Langkah dapat diakses berdasarkan nama (`pipeline.named_steps["scaler"]`), dan parameternya dirujuk dengan format `langkah__param` (misalnya `scaler__with_mean`), sintaks yang dipakai *grid search*.
* `set_config(display="diagram")` membuat Jupyter menampilkan estimator sebagai diagram HTML interaktif. Ini sudah menjadi default sejak v1.1, jadi pemanggilan ini hanya membuatnya eksplisit.

**Asumsi, kelebihan, keterbatasan.**

* *Pipeline* menjamin statistik pra-pemrosesan hanya berasal dari data yang diberikan ke `fit`. Itu pertahanan utama terhadap kebocoran selama pemilihan model.
* *Pipeline* tidak dapat melindungi dari kebocoran yang sudah ada *di dalam fitur*: jika suatu kolom diturunkan dari target, *pipeline* akan tetap memakainya. Resep di bawah secara tidak sengaja menunjukkan hal ini.
* *Pipeline* adalah rantai linear. Percabangan per jenis kolom memerlukan `ColumnTransformer` sebagai salah satu langkahnya.

### Kode

Buku menyusun masalah *supervised* dari tabel hasil encoding Resep 2.3 dengan menjadikan **kolom terakhir sebagai target**. Kolom terakhir itu adalah `Position_Senior`, sehingga tugasnya adalah "memprediksi apakah seorang karyawan berjabatan Senior". Fiturnya masih mencakup `Position_Junior` dan `Position_Manager`, dan karena setiap karyawan memiliki tepat satu jabatan, $\text{Senior} = 1 - \text{Junior} - \text{Manager}$. Dengan kata lain, target dapat dihitung persis dari dua input. Ingat hal ini, karena ini menjelaskan hasil seleksi fitur di Resep 2.5.

In [17]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Fitur = semua kolom kecuali yang terakhir; target = kolom terakhir (Position_Senior)
X = transformed_df.iloc[:, :-1]
y = transformed_df.iloc[:, -1]

# Pembagian 80/20 -> 16 baris latih, 4 baris uji
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2024
)

# Rantai pra-pemrosesan dua langkah: imputasi, lalu standardisasi
pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
    ]
)

# Pelajari rata-rata imputasi dan statistik penskalaan hanya dari baris latih...
X_train_transformed = pipeline.fit_transform(X_train)

# ...lalu pakai ulang statistik yang sama pada baris uji
X_test_transformed = pipeline.transform(X_test)

# Kembalikan array ke DataFrame agar nama kolom dan indeks baris tetap ada
X_train_transformed = pd.DataFrame(
    X_train_transformed, columns=X_train.columns, index=X_train.index
)

X_test_transformed = pd.DataFrame(
    X_test_transformed, columns=X_test.columns, index=X_test.index
)

In [18]:
# Konfirmasi ukuran pembagian dan bahwa targetnya Position_Senior (tambahan)
print("X_train:", X_train_transformed.shape, "| X_test:", X_test_transformed.shape)
print("Kolom target:", y.name)
print("Senior == 1 - Junior - Manager untuk setiap baris:",
      bool(((1 - X["Position_Junior"] - X["Position_Manager"]) == y).all()))

X_train: (16, 9) | X_test: (4, 9)
Kolom target: Position_Senior
Senior == 1 - Junior - Manager untuk setiap baris: True


**Interpretasi output.** Pembagian menghasilkan **16 baris latih dan 4 baris uji**, masing-masing dengan 9 fitur. Pengecekan mengonfirmasi bahwa targetnya adalah `Position_Senior` dan nilainya sama dengan `1 − Junior − Manager` untuk **setiap** baris, sehingga target merupakan fungsi deterministik dari dua kolom input. Tidak ada nilai hilang di sini, sehingga langkah imputer praktis tidak melakukan apa-apa. Meski begitu, tetap merupakan praktik yang baik untuk mempertahankannya agar *pipeline* dapat menangani data tidak lengkap di masa depan.

In [19]:
from sklearn import set_config

# Tampilkan estimator sebagai diagram HTML di output notebook
set_config(display="diagram")

pipeline

Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', StandardScaler())])

**Interpretasi output.** Sel ini menampilkan *pipeline* yang sudah di-fit sebagai diagram: `SimpleImputer` → `StandardScaler`. Di sesi Jupyter yang aktif, kotak-kotaknya dapat dibuka untuk melihat parameter setiap langkah. Di GitHub, HTML interaktif tidak dirender dan hanya teks cadangan `Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', StandardScaler())])` yang tampil, dengan informasi yang sama.

### Poin Penting
* `Pipeline` mem-fit setiap langkah pada output langkah sebelumnya dengan **hanya data latih**, lalu menjalankan ulang langkah yang telah dipelajari pada data baru.
* *Pipeline* berperilaku seperti satu estimator (`fit`, `transform`, `predict`, nama `langkah__param`), dan inilah yang membuat *tuning* dengan *cross-validation* bebas kebocoran.
* *Pipeline* tidak dapat mendeteksi kebocoran target yang tersembunyi di dalam fitur itu sendiri: di sini `Position_Senior` sepenuhnya ditentukan oleh dua input.

## Resep 2.5 — Rekayasa Fitur

**Tujuan.** Membuat fitur baru yang memungkinkan model sederhana menangkap pola non-linear (suku polinomial, *bin* diskret) dan membuang fitur yang tidak membantu (eliminasi rekursif, seleksi berbasis model).

### Teori

**Intuisi.** Model linear hanya dapat menarik garis lurus di ruang yang diberikan kepadanya. Menambahkan suku kuadrat dan interaksi memungkinkannya "melengkung". *Binning* memungkinkan setiap rentang variabel memiliki efeknya sendiri. Namun, terlalu banyak fitur menyebabkan *overfitting* dan pelatihan yang lambat, sehingga *seleksi* fitur menanyakan kepada model input mana yang penting dan hanya mempertahankan input tersebut.

**Matematika.**

* **Fitur polinomial** berderajat $d$ mencakup setiap monomial $x_1^{a_1}x_2^{a_2}\cdots x_n^{a_n}$ dengan $a_1+\dots+a_n\le d$. Termasuk suku bias, jumlah kolom output adalah $\binom{n+d}{d}$. Untuk derajat 2, ini adalah konstanta $1$, setiap $x_j$, setiap $x_j^2$, dan setiap hasil kali $x_jx_k$ ($j<k$).
* **Binning seragam** membagi $[m_j, M_j]$ menjadi $B$ interval berlebar sama dengan batas $m_j + \tfrac{k}{B}(M_j-m_j)$, $k=0,\dots,B$, dan mengganti setiap nilai dengan indeks *bin*-nya $\{0,\dots,B-1\}$.
* **Recursive Feature Elimination (RFE):** fit estimator pada himpunan fitur saat ini $S$, urutkan fitur berdasarkan tingkat kepentingan $|w_j|$ (atau `feature_importances_`), buang yang paling tidak penting, dan ulangi sampai $|S|$ sama dengan ukuran target. Fitur yang dibuang lebih awal mendapat peringkat lebih buruk (lebih tinggi).
* **SelectFromModel:** fit sekali dan pertahankan fitur $j$ jika tingkat kepentingannya melewati ambang, di sini $|w_j| \ge \tfrac{1}{n}\sum_k |w_k|$ (`threshold="mean"`).

**Cara scikit-learn mengimplementasikannya.**

* `PolynomialFeatures(degree=2, interaction_only=False, include_bias=True)`: `fit` hanya menghitung kombinasi kolom output, dan `get_feature_names_out` memberi nama (`a b` untuk hasil kali, `a^2` untuk kuadrat).
* `KBinsDiscretizer(n_bins=5, encode="onehot", strategy="quantile")` adalah nilai default-nya. Resep ini memakai `n_bins=3, encode="ordinal", strategy="uniform"`. `fit` menyimpan `bin_edges_` per fitur.
* `RFE(estimator, n_features_to_select=None, step=1)`: setelah `fit`, `ranking_` bernilai 1 untuk fitur yang dipertahankan dan angka lebih besar untuk fitur yang dieliminasi lebih awal, sedangkan `support_` adalah *mask* boolean.
* `SelectFromModel(estimator, threshold=None, prefit=False)` dengan `prefit=False` mem-fit salinan estimator yang disimpan di `estimator_`. `get_support()` mengembalikan *mask* fitur yang dipertahankan, dan tingkat kepentingan dibaca dari `coef_` (nilai mutlak) atau `feature_importances_`.

**Asumsi, kelebihan, keterbatasan.**

* Ekspansi polinomial tumbuh secara kombinatorial dan menghasilkan kolom yang sangat berkorelasi. Padukan dengan regularisasi (Bab 5).
* *Binning* menukar presisi dengan ketahanan dan non-linearitas. *Bin* yang dipelajari dari data latih harus dipakai ulang pada data uji.
* Seleksi berbasis koefisien mengasumsikan fitur berada pada skala yang sama (karena itu diskalakan dulu) dan tidak stabil ketika fitur berkolinear atau sampelnya sangat kecil, seperti di sini dengan 16 baris.

### Kode

In [20]:
from sklearn.preprocessing import PolynomialFeatures

# Semua suku berderajat <=2: bias, suku linear, kuadrat, dan hasil kali berpasangan
poly = PolynomialFeatures(degree=2)

poly_features = poly.fit_transform(X_train_transformed)
poly_features_df = pd.DataFrame(
    poly_features, columns=poly.get_feature_names_out(X_train_transformed.columns)
)
poly_features_df

,1,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager,...,Department_IT^2,Department_IT Department_Sales,Department_IT Position_Junior,Department_IT Position_Manager,Department_Sales^2,Department_Sales Position_Junior,Department_Sales Position_Manager,Position_Junior^2,Position_Junior Position_Manager,Position_Manager^2
0,1.0,0.834298,-0.867451,-1.559779,-0.577350,-0.480384,2.081666,-0.774597,-0.577350,-1.0,...,4.333333,-1.612452,-1.201850,-2.081666,0.600000,0.447214,0.774597,0.333333,0.577350,1.0
1,1.0,-0.689202,1.282818,-0.150946,-0.577350,2.081666,-0.480384,-0.774597,-0.577350,1.0,...,0.230769,0.372104,0.277350,-0.480384,0.600000,0.447214,-0.774597,0.333333,-0.577350,1.0
2,1.0,0.544107,-1.173357,-1.761041,-0.577350,-0.480384,2.081666,-0.774597,-0.577350,1.0,...,4.333333,-1.612452,-1.201850,2.081666,0.600000,0.447214,-0.774597,0.333333,-0.577350,1.0
3,1.0,-1.342131,0.821525,-0.150946,-0.577350,-0.480384,-0.480384,1.290994,-0.577350,-1.0,...,0.230769,-0.620174,0.277350,0.480384,1.666667,-0.745356,-1.290994,0.333333,0.577350,1.0
4,1.0,1.414679,0.611115,0.654101,-0.577350,-0.480384,-0.480384,1.290994,-0.577350,1.0,...,0.230769,-0.620174,0.277350,-0.480384,1.666667,-0.745356,1.290994,0.333333,-0.577350,1.0
5,1.0,-1.414679,-1.479801,1.257887,1.732051,-0.480384,-0.480384,-0.774597,-0.577350,1.0,...,0.230769,0.372104,0.277350,-0.480384,0.600000,0.447214,-0.774597,0.333333,-0.577350,1.0
6,1.0,1.269583,0.576819,0.452839,1.732051,-0.480384,-0.480384,-0.774597,1.732051,-1.0,...,0.230769,0.372104,-0.832050,0.480384,0.600000,-1.341641,0.774597,3.000000,-1.732051,1.0
7,1.0,0.544107,-1.475450,-0.553470,-0.577350,-0.480384,2.081666,-0.774597,1.732051,-1.0,...,4.333333,-1.612452,3.605551,-2.081666,0.600000,-1.341641,0.774597,3.000000,-1.732051,1.0
8,1.0,0.906845,1.060444,1.459148,-0.577350,-0.480384,-0.480384,1.290994,1.732051,-1.0,...,0.230769,-0.620174,-0.832050,0.480384,1.666667,2.236068,-1.290994,3.000000,-1.732051,1.0
9,1.0,0.906845,1.664674,0.855363,-0.577350,-0.480384,-0.480384,1.290994,-0.577350,-1.0,...,0.230769,-0.620174,0.277350,0.480384,1.666667,-0.745356,-1.290994,0.333333,0.577350,1.0


**Interpretasi output.** Sembilan fitur input berkembang menjadi **55 kolom** (tampil sebagai "16 rows × 55 columns"), tepat $\binom{9+2}{2}=55$: 1 kolom bias bernilai 1.0, 9 suku linear, 9 kuadrat, dan 36 hasil kali berpasangan. Beberapa kolom baru ini redundan. Misalnya, `Position_Manager^2` bernilai 1.0 di setiap baris: di dalam 16 baris latih, `Position_Manager` hanya memiliki dua nilai yang setelah distandardisasi menjadi tepat ±1, sehingga kuadratnya selalu 1. Ekspansi seperti ini memerlukan regularisasi atau seleksi sesudahnya.

In [21]:
from sklearn.preprocessing import KBinsDiscretizer

# 3 bin berlebar sama per fitur, dikembalikan sebagai id bin bilangan bulat 0/1/2
kbins = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="uniform")

binned_data = kbins.fit_transform(X_train_transformed)
binned_df = pd.DataFrame(binned_data, columns=X_train_transformed.columns)
binned_df

,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager
0,2.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0
1,0.0,2.0,1.0,0.0,2.0,0.0,0.0,0.0,2.0
2,2.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,2.0
3,0.0,2.0,1.0,0.0,0.0,0.0,2.0,0.0,0.0
4,2.0,1.0,2.0,0.0,0.0,0.0,2.0,0.0,2.0
5,0.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,2.0
6,2.0,1.0,2.0,2.0,0.0,0.0,0.0,2.0,0.0
7,2.0,0.0,1.0,0.0,0.0,2.0,0.0,2.0,0.0
8,2.0,2.0,2.0,0.0,0.0,0.0,2.0,2.0,0.0
9,2.0,2.0,2.0,0.0,0.0,0.0,2.0,0.0,0.0


**Interpretasi output.** Setiap nilai diganti dengan id *bin*-nya (0 = sepertiga bawah, 1 = sepertiga tengah, 2 = sepertiga atas dari rentang kolom tersebut). `Salary_scaled` dan `Experience_scaled` memakai ketiga *bin*, sedangkan `Age_scaled` hanya menunjukkan *bin* 0 dan 2: tidak ada satu pun dari 16 usia latih yang jatuh di sepertiga tengah rentang usia. **Kolom one-hot hanya pernah menempati bin 0 dan 2**: kolom biner hanya memiliki dua nilai berbeda yang berada di kedua ujung rentangnya, sehingga *bin* tengah selalu kosong. Mendiskretkan fitur biner tidak menambah apa pun; dalam praktiknya hanya kolom kontinu yang perlu didiskretkan.

In [22]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

# Eliminasi satu fitur per putaran sampai tersisa satu fitur
rfe = RFE(estimator=LinearRegression(), n_features_to_select=1)

rfe.fit(X_train_transformed, y_train)

# ranking_: 1 = yang bertahan terakhir, angka lebih besar = dieliminasi lebih awal
rfe.ranking_

array([8, 7, 9, 5, 6, 4, 3, 2, 1])

In [23]:
# Pasangkan nama fitur dengan peringkatnya agar mudah dibaca (tambahan)
pd.Series(rfe.ranking_, index=X_train_transformed.columns).sort_values()

Position_Manager      1
Position_Junior       2
Department_Sales      3
Department_IT         4
Department_Finance    5
Department_HR         6
Salary_scaled         7
Age_scaled            8
Experience_scaled     9
dtype: int64

**Interpretasi output.** Peringkat mentah `[8, 7, 9, 5, 6, 4, 3, 2, 1]` menjadi jelas setelah nama dipasangkan: **`Position_Manager` berperingkat 1 dan `Position_Junior` 2**, diikuti kolom departemen, sedangkan `Experience_scaled` (9), `Age_scaled` (8), dan `Salary_scaled` (7) dieliminasi pertama kali. Inilah kebocoran target dari Resep 2.4 yang muncul: karena Senior = 1 − Junior − Manager, dua kolom jabatan menjelaskan target dengan sempurna dan sisanya hanyalah *noise*.

In [24]:
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LinearRegression

# Pertahankan fitur yang |koefisien|-nya minimal rata-rata |koefisien|
selector = SelectFromModel(
    estimator=LinearRegression(),
    prefit=False,
    threshold="mean",
)

selector.fit(X_train_transformed, y_train)

# Mask boolean fitur yang dipertahankan, lalu nama-namanya
selected_features_mask = selector.get_support()
selected_features = X_train_transformed.columns[selected_features_mask].tolist()

# Koefisien setiap fitur dan apakah lolos ambang
feature_importance = pd.DataFrame(
    {
        "Feature": X_train_transformed.columns,
        "Importance": selector.estimator_.coef_,
        "Selected": selected_features_mask,
    }
)
feature_importance.sort_values("Importance", key=abs, ascending=False)

,Feature,Importance,Selected
8,Position_Manager,-5.000000e-01,True
7,Position_Junior,-4.330127e-01,True
5,Department_IT,8.560455e-16,False
6,Department_Sales,6.643452e-16,False
4,Department_HR,4.907588e-16,False
3,Department_Finance,4.611122e-16,False
0,Age_scaled,-2.727983e-16,False
1,Salary_scaled,1.817895e-16,False
2,Experience_scaled,-5.133914e-17,False


**Interpretasi output.** Hanya **`Position_Manager` (koefisien −0.500)** dan **`Position_Junior` (−0.433)** yang melewati ambang rata-rata |koefisien|. Semua koefisien lain berorde $10^{-16}$, yang secara numerik sama dengan nol (galat pembulatan *floating-point*). Tanda negatifnya masuk akal: menjadi Manager atau Junior berarti *bukan* Senior. Kedua koefisien berbeda (−0.500 vs −0.433) hanya karena masing-masing dinyatakan dalam satuan kolom terstandardisasinya sendiri. Jika diskalakan kembali ke kolom 0/1 asli, keduanya sama dengan −1, sehingga persis memulihkan Senior = 1 − Junior − Manager. Dalam proyek nyata, hasil seperti ini adalah tanda bahaya kebocoran, bukan tanda model yang hebat.

### Poin Penting
* `PolynomialFeatures` menumbuhkan fitur sebanyak $\binom{n+d}{d}$ dan menghasilkan kolom redundan (kuadrat fitur biner), jadi lakukan regularisasi sesudahnya.
* `KBinsDiscretizer` berguna untuk variabel kontinu. Kolom biner hanya menempati *bin* ekstrem.
* `RFE` dan `SelectFromModel` sama-sama memilih dua kolom jabatan, yang dengan tepat mengungkap bahwa target dibangun dari kolom tersebut.
* Seleksi fitur hanya dapat dipercaya sebatas fitur-fiturnya: selalu periksa kebocoran sebelum merayakan fitur yang "sempurna".

## Resep 2.6 — Latihan Praktis Pra-pemrosesan Data

**Tujuan.** Menggabungkan gagasan-gagasan bab ini menjadi satu `Pipeline` menyeluruh (imputasi → penskalaan → model) dan mengevaluasinya pada dataset nyata, California Housing.

### Teori

**Intuisi.** Proyek nyata membungkus pra-pemrosesan dan model dalam satu objek agar transformasi yang persis sama diterapkan saat pelatihan, evaluasi, dan produksi. Dataset ini berisi 20.640 distrik di California (sensus 1990) dengan 8 fitur numerik (median pendapatan, usia rumah, rata-rata jumlah kamar, populasi, lintang/bujur, …). Targetnya adalah median nilai rumah dalam satuan \$100.000.

**Matematika.** Langkah terakhir, `RandomForestRegressor`, merata-ratakan $B$ pohon regresi yang dilatih pada sampel *bootstrap*: $\hat{y}(\mathbf{x}) = \frac{1}{B}\sum_{b=1}^{B} T_b(\mathbf{x})$ (rinciannya di Bab 8). `pipeline.score` mengembalikan koefisien determinasi
$$R^2 = 1-\frac{\sum_i (y_i-\hat{y}_i)^2}{\sum_i (y_i-\bar{y})^2},$$
yaitu proporsi variansi target yang dijelaskan model ($1$ = sempurna, $0$ = tidak lebih baik daripada memprediksi rata-rata $\bar{y}$).

**Cara scikit-learn mengimplementasikannya.** `fetch_california_housing()` mengunduh data sekali dan menyimpannya di cache `~/scikit_learn_data`, sehingga tidak ada data yang disimpan di repositori ini. *Pipeline* menjalankan `SimpleImputer(strategy="median")` → `StandardScaler()` → `RandomForestRegressor()` dengan parameter default (`n_estimators=100`, `max_features=1.0`, pohon tumbuh penuh). Memanggil `score(X_test, y_test)` mentransformasi data uji dengan statistik data latih lalu mengembalikan $R^2$.

**Asumsi, kelebihan, keterbatasan.** Dataset ini tidak memiliki nilai hilang dan *random forest* tidak terpengaruh skala, sehingga imputer dan scaler tidak mengubah prediksi di sini. Keduanya disertakan untuk menunjukkan polanya. *Forest* tidak diberi `random_state`, sehingga hasilnya dapat direproduksi hanya karena sel persiapan memberi seed pada generator global NumPy.

### Kode

In [25]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# Unduh (hanya pada eksekusi pertama) dan muat data California Housing
california_data = fetch_california_housing()
X, y = california_data.data, california_data.target

# Sisihkan 20% distrik untuk pengujian
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2024
)

# Seluruh alur kerja dalam satu objek: imputasi -> penskalaan -> random forest
comprehensive_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", RandomForestRegressor()),
    ]
)

comprehensive_pipeline.fit(X_train, y_train)

# Pipeline.score diteruskan ke regresor -> R^2 pada data uji
score = comprehensive_pipeline.score(X_test, y_test)
score

0.8026768047828426

In [26]:
# Konteks untuk skor: ukuran data dan nilai hilang (tambahan)
print("Sampel x fitur     :", X.shape)
print("Baris latih / uji  :", X_train.shape[0], "/", X_test.shape[0])
print("Nilai hilang       :", int(np.isnan(X).sum()))

Sampel x fitur     : (20640, 8)
Baris latih / uji  : 16512 / 4128
Nilai hilang       : 0


**Interpretasi output.** *Pipeline* mencapai **$R^2 \approx 0{,}803$** pada 4.128 distrik yang disisihkan (dilatih pada 16.512): *forest* menjelaskan sekitar 80 % variansi median nilai rumah. Sel konteks mengonfirmasi bahwa dataset memiliki **0 nilai hilang**, sehingga imputer median hanyalah jaring pengaman, bukan langkah yang benar-benar bekerja. Langkah selanjutnya adalah memakai *cross-validation* alih-alih satu kali pembagian, serta *tuning* hiperparameter (Bab 8).

### Poin Penting
* Alur kerja lengkap (pra-pemrosesan + model) muat dalam satu `Pipeline` dan dievaluasi dengan satu panggilan `score`.
* *Random forest* default menjelaskan sekitar 80 % variansi nilai rumah pada distrik yang belum pernah dilihat.
* Langkah yang tidak berpengaruh untuk data saat ini (imputer, scaler) tetap membuat *pipeline* tahan terhadap data masa depan dan memudahkan penggantian model.

## Ringkasan Bab

| Resep | Konsep | Kelas/fungsi sklearn utama | Poin penting |
|---|---|---|---|
| 2.1 Menangani Data Hilang | Imputasi (statistik kolom, tetangga terdekat, regresi iteratif) | `SimpleImputer`, `KNNImputer`, `IterativeImputer` | Imputer yang lebih canggih hanya membantu jika fitur saling membawa informasi; pada kolom independen hasil iteratif ≈ rata-rata. |
| 2.2 Teknik Penskalaan | Standardisasi, penskalaan min-max, normalisasi baris | `StandardScaler`, `MinMaxScaler`, `Normalizer` | Scaler kolom memperbaiki satuan per fitur; `Normalizer` menskalakan baris dan punya tujuan berbeda. |
| 2.3 Encoding Variabel Kategorikal | One-hot vs kode bilangan bulat, routing per kolom | `OneHotEncoder`, `LabelEncoder`, `ColumnTransformer` | One-hot menghindari urutan palsu; `ColumnTransformer` menggabungkan pra-pemrosesan numerik dan kategorikal. |
| 2.4 Pengantar *Pipeline* | Menyusun transformer, fitting bebas kebocoran | `Pipeline`, `train_test_split`, `set_config` | Fit di data latih, terapkan ulang di data uji; *pipeline* tidak dapat menghapus kebocoran yang sudah ada di fitur. |
| 2.5 Rekayasa Fitur | Ekspansi polinomial, binning, seleksi fitur | `PolynomialFeatures`, `KBinsDiscretizer`, `RFE`, `SelectFromModel` | 9 → 55 fitur dengan derajat 2; kedua selector mengungkap kolom `Position_*` yang bocor. |
| 2.6 Latihan Praktis | Pra-pemrosesan + model menyeluruh | `Pipeline`, `RandomForestRegressor`, `fetch_california_housing` | Satu objek menangani seluruh alur kerja; R² ≈ 0,80 pada distrik yang disisihkan. |

**Kaitan dengan Bab 3.** Bab 2 diakhiri dengan membuat fitur yang *lebih banyak* (55 kolom polinomial dari 9). Bab 3 menempuh arah sebaliknya: **reduksi dimensi** (PCA, LDA, t-SNE) memampatkan banyak fitur yang saling berkorelasi menjadi sedikit arah yang informatif. PCA khususnya bergantung pada pola `StandardScaler` + `Pipeline` yang dibangun di bab ini.

## Referensi

* Sukup, J. (2025). *scikit-learn Cookbook, Third Edition*. Packt Publishing. Bab 2.
* Repositori kode resmi: <https://github.com/PacktPublishing/Scikit-Learn-Cookbook-Third-Edition>
* scikit-learn User Guide — Imputation of missing values: <https://scikit-learn.org/1.5/modules/impute.html>
* scikit-learn User Guide — Preprocessing data (scaling, normalization, encoding, discretization): <https://scikit-learn.org/1.5/modules/preprocessing.html>
* scikit-learn User Guide — Pipelines and composite estimators: <https://scikit-learn.org/1.5/modules/compose.html>
* scikit-learn User Guide — Feature selection: <https://scikit-learn.org/1.5/modules/feature_selection.html>
* scikit-learn API — `PolynomialFeatures`: <https://scikit-learn.org/1.5/modules/generated/sklearn.preprocessing.PolynomialFeatures.html>
* scikit-learn datasets — California Housing: <https://scikit-learn.org/1.5/datasets/real_world.html#california-housing-dataset>